In [1]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
load_dotenv()

True

In [3]:
llm = init_chat_model(model="groq:qwen/qwen3.8-27b")

In [4]:
tavily_client = TavilyClient()

@tool
def tavily_search(query: str) -> dict:
    """Search the web for up-to-date information, news, facts, and answers."""
    return tavily_client.search(query=query)


In [5]:
#custom fucntion
@tool
def multiply(a:int, b:int) -> int:
    """multiply a and b
    
    args:
    a: first int
    b: second int
    
    returns:
    int: output int"""

In [6]:
tools = [tavily_search, multiply]
llm_with_tools = llm.bind_tools(tools)
 

In [7]:
llm_with_tools 

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x0000019AA9F5A0C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019AA9FA4110>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for up-to-date information, news, facts, and answers.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply a and b\n\n    args:\n    a: first int\n    b: second int\n\n    returns:\n    int: output int', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [9]:
#node functionality
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state['messages'])]}

In [10]:
#graph
builder =StateGraph(State)

In [13]:
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools",ToolNode(tools))